# Pill Defect Detection via Diffusion-Based Synthetic Data

Colab version of the pipeline (the README shows the same steps for a local GPU).

**Runtime → Change runtime type → T4 GPU.** Put in Google Drive under `MyDrive/pill_defect/`:
- `pill.tar.xz` (MVTec AD, pill)
- the project folder `pill-defect-synthetic/`

All outputs go to `MyDrive/pill_defect/work/`, so a disconnect loses nothing. Generation resumes where it stopped (run `repair_partial.py` first).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/pill_defect'
PROJ = f'{ROOT}/pill-defect-synthetic'
WORK = f'{ROOT}/work'
DATA = '/content/data'
SYN = f'{WORK}/pill_defects'
RUNS = f'{WORK}/runs'
%cd {PROJ}/src
!mkdir -p {WORK} {DATA} {RUNS}

In [ ]:
!pip -q install diffusers transformers accelerate peft opencv-python-headless pandas
# Colab ships an old torchao that makes recent peft fail on import; it is not needed here
!pip -q uninstall -y torchao

## 1. Data (~2 min)

In [ ]:
!tar -xJf {ROOT}/pill.tar.xz -C {DATA}
!git clone -q --depth 1 https://github.com/matlab-deep-learning/pillQC.git {DATA}/pillQC
import os
# expected: train/good 267 | test/good 26, crack 26, scratch 24, contamination 21
for d in ['train/good', 'test/good', 'test/crack', 'test/scratch', 'test/contamination']:
    print(d, len(os.listdir(f'{DATA}/pill/{d}')))

## 2. PillQC chip masks + DefectFill LoRA (~60–80 min for 2000 steps)
PillQC has no masks; they are rebuilt from the pill outline. Check the preview first.

In [ ]:
!python make_chip_masks.py --pillqc {DATA}/pillQC --out {WORK}/pillqc_chip
from IPython.display import Image, display
display(Image(f'{WORK}/pillqc_chip/preview.jpg'))

In [ ]:
!python train_lora_defectfill.py --data {WORK}/pillqc_chip --out {WORK}/defectfill_lora_weights --steps 2000

## 3. Trial generation: 6 per class (~3 min)
Look before the long run.

`--crack-masks edge` (default) keeps the test set unseen. `--crack-masks mvtec_gt` reproduces the original, which shaped cracks with MVTec **test** masks.

In [ ]:
!python generate_defects.py --mvtec {DATA}/pill --lora {WORK}/defectfill_lora_weights --out {WORK}/trial --test 6 --crack-masks edge
!python clean.py sheets --syn {WORK}/trial
import glob
for f in sorted(glob.glob(f'{WORK}/trial/sheets/*.jpg')):
    print(f); display(Image(f))

## 4. Full generation (~2,800 inpaints + 1,000 good; roughly 3–4 h on a T4, about 1 h on an RTX 5070 Ti)
The reported results used `--crack-masks mvtec_gt` (crack shapes from MVTec test masks; the README reports AUROC without cracks too). Use `edge` to keep the test set fully unseen.

After a disconnect: run `repair_partial.py --apply`, then this cell again.

In [ ]:
!python repair_partial.py --syn {SYN} --apply
!python generate_defects.py --mvtec {DATA}/pill --lora {WORK}/defectfill_lora_weights --out {SYN} --crack-masks mvtec_gt

## 5. Quality control
1. `auto` removes inpaints that barely changed the pill.
2. `qc_rank.py` scores every image and writes review sheets of the least visible 15% per class (`work/pill_defects/qc/suspects_*.jpg`: full image + 2× zoom).
3. Write the bad ones into `work/blacklist.txt` (one per line, e.g. `crack_012.png`); `apply` moves them to `rejected/`.

In [ ]:
!python clean.py auto --syn {SYN} --mvtec {DATA}/pill --min-change 6
!python qc_rank.py --syn {SYN} --mvtec {DATA}/pill

In [ ]:
!touch {WORK}/blacklist.txt
!python clean.py apply --syn {SYN} --blacklist {WORK}/blacklist.txt

## 6. Baselines
Baseline 3 `v4_vae` needs the good images passed through the SD VAE first (removes the generation shortcut; see README). Versions with seeds run 3 times.

In [ ]:
!python baseline_1_autoencoder.py --mvtec {DATA}/pill --out {RUNS}/b1_autoencoder

In [ ]:
!python baseline_2_synthetic_multiclass.py --syn {SYN} --out {RUNS}/b2_random_split
!python baseline_2_synthetic_multiclass.py --syn {SYN} --out {RUNS}/b2_group_split --group-split

In [ ]:
!python vae_roundtrip.py --src {DATA}/pill/train/good --dst {SYN}/good_vae/images --prefix real_
!python vae_roundtrip.py --src {SYN}/good/images --dst {SYN}/good_vae/images --prefix syn_
for v in ['v1', 'v2']:
    !python baseline_3_binary.py --version {v} --mvtec {DATA}/pill --syn {SYN} --out {RUNS}/b3_{v}
for v in ['v4_vae', 'v4_vae_aug', 'v3_aug', 'v3_aug_nosyngood']:
    for seed in [1, 2, 3]:
        !python baseline_3_binary.py --version {v} --mvtec {DATA}/pill --syn {SYN} --out {RUNS}/b3_{v}_s{seed} --seed {seed}

In [ ]:
# Shortcut test: do real good pills get flagged once they carry the VAE fingerprint?
!python diagnose_shortcut.py --mvtec {DATA}/pill --runs {RUNS}/b3_v1 {RUNS}/b3_v2 {RUNS}/b3_v4_vae_s1

## 7. Results table
Read AUROC first. `flag-all F1` is what calling every pill defective scores (~0.845 on the targeted set). F1 is at threshold 0.5; `best F1` uses a threshold chosen on the test set, so it is optimistic.

In [ ]:
import json, pandas as pd
rows = []
b1 = json.load(open(f'{RUNS}/b1_autoencoder/eval.json'))
for ts in ['targeted', 'all']:
    r = b1[ts]; t = r['at_train_p95_threshold']
    rows.append({'model': 'B1 autoencoder', 'test': ts, 'seeds': 1, 'AUROC': r['auroc'], 'F1': t['f1'], 'P': t['precision'], 'R': t['recall']})
runs = {'v1': ['b3_v1'], 'v2': ['b3_v2']}
runs.update({v: [f'b3_{v}_s{s}' for s in (1, 2, 3)] for v in ['v3_aug', 'v3_aug_nosyngood', 'v4_vae', 'v4_vae_aug']})
for v, dirs in runs.items():
    for ts in ['targeted', 'all']:
        rs = [json.load(open(f'{RUNS}/{d}/eval.json'))[ts] for d in dirs]
        s = lambda f: pd.Series([f(r) for r in rs])
        fmt = lambda x: f'{x.mean():.3f} ± {x.std():.3f}' if len(x) > 1 else f'{x.iloc[0]:.3f}'
        rows.append({'model': f'B3 {v}', 'test': ts, 'seeds': len(rs), 'AUROC': fmt(s(lambda r: r['auroc'])),
                     'F1': fmt(s(lambda r: r['default']['f1'])), 'P': fmt(s(lambda r: r['default']['precision'])),
                     'R': fmt(s(lambda r: r['default']['recall']))})
print('B2 random split:', json.load(open(f'{RUNS}/b2_random_split/eval.json'))['macro_f1'], '| group split:', json.load(open(f'{RUNS}/b2_group_split/eval.json'))['macro_f1'])
pd.DataFrame(rows)